In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [ ]:
df = pd.read_csv('/vol/miltank/users/doanb/ADLM_SS2025/explore_ROCO_dataset/ROCO-dataset/radiologytraindata.csv')
#df.columns # Index(['id', 'name', 'caption'], dtype='object')
df.head() 

,id,name,caption
0,ROCO_00002,PMC4083729_AMHSR-4-14-g002.jpg,Computed tomography scan in axial view showin...
1,ROCO_00003,PMC2837471_IJD2009-150251.001.jpg,Bacterial contamination occurred after comple...
2,ROCO_00004,PMC2505281_11999_2007_30_Fig6_HTML.jpg,The patient had residual paralysis of the han...
3,ROCO_00005,PMC3745845_IJD2013-683423.005.jpg,Panoramic radiograph after immediate loading.\n
4,ROCO_00007,PMC4917066_amjcaserep-17-301-g001.jpg,Plain abdomen x-ray: Multiple air levels at t...


In [3]:
# print caption for name "PMC3563702_trd-74-37-g001.jpg"
caption = df[df['name'] == 'PMC3563702_trd-74-37-g001.jpg']['caption'].values[0]
print(caption)

 Chest computed tomography scan showed a mass lesion on the upper lobe of the right lung.



In [ ]:
# Filter only entries with 'caption' containing 'chest' and "mediasti"
df_chest = df[df['caption'].str.contains('liver', case=False, na=False) #chest / abomen|abdominal
              & df['caption'].str.contains('enlarged', case=False, na=False) # mediasti/ lung / cardiac|cardiomegaly
              #& df['caption'].str.contains('nodul|mass', case=False, na=False)
            ]
print(f"Number of entries: {len(df_chest)}")

# save the column 'name' to a new csv file 
df_chest[['name']].to_csv('/vol/miltank/users/doanb/ADLM_SS2025/ngocs_use_cases/filtered_chest_names.csv', index=False)

df_chest.to_excel('/vol/miltank/users/doanb/ADLM_SS2025/ngocs_use_cases/filtered_chest.xlsx', index=False)

df_chest.head()



Number of entries: 165


,id,name,caption
232,ROCO_00292,PMC5359795_10.1177_2055116915579680-fig1.jpg,Right (R) lateral recumbent radiograph of the...
676,ROCO_00851,PMC4974773_13104_2016_2181_Fig3_HTML.jpg,Chest radiograph showing cardiomegaly and bil...
1242,ROCO_01555,PMC4352732_CRIC2015-319312.002.jpg,"CXR done on 2nd day shows pulmonary edema, ca..."
1466,ROCO_01838,PMC3669303_pone.0064603.g003.jpg,Postero-anterior chest radiograph of the prob...
1555,ROCO_01962,PMC4371810_13104_2015_1031_Fig2_HTML.jpg,CXR showed gross cardiomegaly and bilateral ...


In [7]:
not_include_terms = [
    "ct", "computed", "tomography","MRI", "MR", "magnetic", "resonance", "X-ray", "radiograph", "PET", 
    "scan", "scans", "image", "images", "figure", "figures", 
    "shows", "showing", "seen", "demonstrating", "presenting", "demonstrates",
]
default_english_stop_words = CountVectorizer(stop_words='english').get_stop_words()
custom_stop_words = list(set(default_english_stop_words).union(term.lower() for term in not_include_terms))

# Extract common n-grams
vectorizer = CountVectorizer(ngram_range=(1, 4),  stop_words=custom_stop_words, min_df=5) 
X = vectorizer.fit_transform(df_chest['caption'])
# Get the feature names (bigrams and trigrams)
feature_names = vectorizer.get_feature_names_out()
# Convert to a DataFrame for better visualization
df_bigrams_trigrams = pd.DataFrame(X.toarray(), columns=feature_names)
# Sum the occurrences of each bigram and trigram
bigram_trigram_counts = df_bigrams_trigrams.sum().sort_values(ascending=False)

# save to an excel file
output_file = '/vol/miltank/users/doanb/ADLM_SS2025/ngocs_use_cases/n_grams_lung.xlsx'
bigram_trigram_counts.to_excel(output_file, index=True, header=['Count'])

# make bigram_trigram_counts to a DataFrame
#df_bigrams_trigrams_counts = pd.DataFrame(bigram_trigram_counts).reset_index()
#df_bigrams_trigrams_counts.columns = ['n_gram', 'count']

/meta/opt/anaconda3/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ray'] not in stop_words.
  warnings.warn(


In [15]:
# print out the caption of the image with name 'PMC529310_1479-5876-2-33-1.jpg'
print(df[df['name'] == 'PMC529310_1479-5876-2-33-1.jpg']['caption'].values[0])

 Chest CT before treatment (27-Sep-2001) show that conglomeration of a size of 5.5 × 4.2 cm at the left lower hilus pulmonis, large amount of accumulation of fluid in the left thoracic cavity, enlarged lymph nodes in the mediastinum.

